# SalesLT - Gold analytics

**Medallion role:** Gold. This notebook derives product, customer, and monthly sales summaries from the enriched Silver sales-order-line fact. The outputs provide stable analytical grains, consistent financial measures, and reusable rankings for reporting.

In [0]:
dbutils.widgets.removeAll()

## Analytical contract

The environment widget resolves the Silver source and external Gold targets. Product, customer, and month keys define independent refresh grains while preserving a single upstream source of truth.

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()

ingestion_timestamp = (
    dbutils.widgets.get("ingestion_timestamp").strip()
)

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "saleslt_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "saleslt_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

sales_silver = (
    f"{catalog}.silver.sales_order_lines"
)

product_gold = (
    f"{catalog}.gold.sales_by_product"
)

customer_gold = (
    f"{catalog}.gold.sales_by_customer"
)

monthly_gold = (
    f"{catalog}.gold.monthly_sales_summary"
)

product_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "saleslt/gold/sales_by_product/"
)

customer_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "saleslt/gold/sales_by_customer/"
)

monthly_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "saleslt/gold/monthly_sales_summary/"
)

print("=" * 60)
print("SALESLT - GOLD ANALYTICS")
print("=" * 60)
print(f"Environment       : {environment}")
print(f"Source            : {sales_silver}")
print(f"Product target    : {product_gold}")
print(f"Customer target   : {customer_gold}")
print(f"Monthly target    : {monthly_gold}")
print("=" * 60)

## Product, customer, and time-series measures

The aggregations calculate order and customer reach, units, revenue, discounts, spending, and order-value statistics from normalized Silver amounts. Product ranking is derived after aggregation, and monthly summaries provide a calendar grain suited to trend analysis.

In [0]:
product_sales_df = spark.sql(f"""
SELECT
    product_id,

    product_name,

    product_category,

    COUNT(DISTINCT sales_order_id)
        AS total_orders,

    COUNT(DISTINCT customer_id)
        AS total_customers,

    SUM(order_quantity)
        AS total_units_sold,

    CAST(
        SUM(gross_line_amount)
        AS DECIMAL(18,2)
    ) AS gross_revenue,

    CAST(
        SUM(discount_amount)
        AS DECIMAL(18,2)
    ) AS total_discount,

    CAST(
        SUM(net_line_amount)
        AS DECIMAL(18,2)
    ) AS net_revenue,

    CAST(
        AVG(net_line_amount)
        AS DECIMAL(18,2)
    ) AS average_line_value,

    CAST(
        MIN(net_line_amount)
        AS DECIMAL(18,2)
    ) AS min_line_value,

    CAST(
        MAX(net_line_amount)
        AS DECIMAL(18,2)
    ) AS max_line_value,

    CAST(
        '{ingestion_timestamp}'
        AS TIMESTAMP
    ) AS gold_processing_timestamp

FROM {sales_silver}

GROUP BY
    product_id,
    product_name,
    product_category
""")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    dense_rank
)

product_rank_window = (
    Window.orderBy(
        col("net_revenue").desc()
    )
)

product_sales_df = (
    product_sales_df
        .withColumn(
            "revenue_rank",
            dense_rank().over(
                product_rank_window
            )
        )
)

In [0]:
customer_sales_df = spark.sql(f"""
SELECT
    customer_id,

    customer_name,

    customer_email,

    COUNT(DISTINCT sales_order_id)
        AS total_orders,

    COUNT(DISTINCT product_id)
        AS distinct_products,

    SUM(order_quantity)
        AS total_units_purchased,

    CAST(
        SUM(gross_line_amount)
        AS DECIMAL(18,2)
    ) AS gross_revenue,

    CAST(
        SUM(discount_amount)
        AS DECIMAL(18,2)
    ) AS total_discount,

    CAST(
        SUM(net_line_amount)
        AS DECIMAL(18,2)
    ) AS total_spent,

    CAST(
        AVG(net_line_amount)
        AS DECIMAL(18,2)
    ) AS average_line_value,

    MIN(order_date)
        AS first_order_date,

    MAX(order_date)
        AS last_order_date,

    CAST(
        '{ingestion_timestamp}'
        AS TIMESTAMP
    ) AS gold_processing_timestamp

FROM {sales_silver}

GROUP BY
    customer_id,
    customer_name,
    customer_email

HAVING
    COUNT(DISTINCT sales_order_id) >= 1
""")

In [0]:
monthly_sales_df = spark.sql(f"""
SELECT
    YEAR(order_date)
        AS sales_year,

    MONTH(order_date)
        AS sales_month,

    DATE_TRUNC(
        'MONTH',
        order_date
    ) AS month_start_date,

    COUNT(DISTINCT sales_order_id)
        AS total_orders,

    COUNT(DISTINCT customer_id)
        AS total_customers,

    COUNT(DISTINCT product_id)
        AS distinct_products,

    SUM(order_quantity)
        AS total_units_sold,

    CAST(
        SUM(gross_line_amount)
        AS DECIMAL(18,2)
    ) AS gross_revenue,

    CAST(
        SUM(discount_amount)
        AS DECIMAL(18,2)
    ) AS total_discount,

    CAST(
        SUM(net_line_amount)
        AS DECIMAL(18,2)
    ) AS net_revenue,

    CAST(
        AVG(net_line_amount)
        AS DECIMAL(18,2)
    ) AS average_line_value,

    CAST(
        MIN(net_line_amount)
        AS DECIMAL(18,2)
    ) AS min_line_value,

    CAST(
        MAX(net_line_amount)
        AS DECIMAL(18,2)
    ) AS max_line_value,

    CAST(
        '{ingestion_timestamp}'
        AS TIMESTAMP
    ) AS gold_processing_timestamp

FROM {sales_silver}

GROUP BY
    YEAR(order_date),
    MONTH(order_date),
    DATE_TRUNC(
        'MONTH',
        order_date
    )
""")

## Synchronized Gold snapshots

Each target is created as an external Delta table on its first run and MERGEd on subsequent runs using its complete business key. Updates, inserts, and removal of groups no longer present in Silver keep reporting tables aligned with the latest source snapshot.

In [0]:
# COMMAND ----------

from delta.tables import DeltaTable


def merge_gold_snapshot(
    source_df,
    target_table,
    target_path,
    merge_condition
):
    """
    Creates an external Gold Delta table on initial load.

    Subsequent executions synchronize the Gold table with
    the latest analytical snapshot using Delta MERGE.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating Gold table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option(
                    "mergeSchema",
                    "true"
                )
                .option(
                    "path",
                    target_path
                )
                .saveAsTable(
                    target_table
                )
        )

    else:

        print(
            f"Synchronizing Gold table: "
            f"{target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")

                .merge(
                    source_df.alias("source"),
                    merge_condition
                )

                .withSchemaEvolution()

                .whenMatchedUpdateAll()

                .whenNotMatchedInsertAll()

                .whenNotMatchedBySourceDelete()

                .execute()
        )

    print(
        f"Gold synchronization completed: "
        f"{target_table}"
    )

In [0]:
merge_gold_snapshot(
    source_df=product_sales_df,
    target_table=product_gold,
    target_path=product_path,
    merge_condition=(
        "target.product_id = source.product_id"
    )
)

merge_gold_snapshot(
    source_df=customer_sales_df,
    target_table=customer_gold,
    target_path=customer_path,
    merge_condition=(
        "target.customer_id = source.customer_id"
    )
)

merge_gold_snapshot(
    source_df=monthly_sales_df,
    target_table=monthly_gold,
    target_path=monthly_path,
    merge_condition=(
        "target.sales_year = source.sales_year "
        "AND target.sales_month = source.sales_month"
    )
)